# 01 — Ingest

**Source: IGDB — Internet Game Database (API).** IGDB (owned by Twitch/Amazon) is a large, actively-maintained game database queried through its v4 API — **not scraped**. Each game can carry a `aggregated_rating` (external **critic** aggregate, 0–100), a `rating` (IGDB **user/community** rating, 0–100), release date, genres and platforms.

**Why IGDB (not Metacritic/RAWG).** Metacritic aggregates a score only slowly and selectively, so recent years look artificially sparse. IGDB's critic coverage keeps up with current releases, so the story stays up to date. The metric is therefore *IGDB critic rating*, not *Metacritic*.

**Auth.** IGDB uses a short-lived OAuth token minted from a free **Twitch** app (client-credentials flow). `TWITCH_CLIENT_ID` + `TWITCH_CLIENT_SECRET` live in the **gitignored `.env`** and are exchanged for a token here — never hardcoded, committed, or printed. Register an app at <https://dev.twitch.tv/console/apps>.

**What we keep.** Games with a critic aggregate backed by **≥ 3 critic scores** (`aggregated_rating_count >= 3`) so a lone review doesn't create a noisy "aggregate." Raw JSON pages are cached to `data/raw/igdb/`; the flattened table lands in DuckDB as `igdb_games_raw` and is registered in `_sources`.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
from src.ingest import load_config, igdb_access_token, fetch_igdb_games, _igdb_id_name_map
from src.clean_quality import get_connection, load_to_duckdb, register_source

load_dotenv('.env')
CLIENT_ID = os.getenv('TWITCH_CLIENT_ID')
CLIENT_SECRET = os.getenv('TWITCH_CLIENT_SECRET')
assert CLIENT_ID and CLIENT_SECRET, 'TWITCH_CLIENT_ID / TWITCH_CLIENT_SECRET missing from .env — register a free app at https://dev.twitch.tv/console/apps'

cfg = load_config('config.yaml')
con = get_connection(cfg)

# Mint a short-lived IGDB OAuth token from the Twitch app credentials.
token = igdb_access_token(CLIENT_ID, CLIENT_SECRET, cfg)
print('Project:', cfg['project_name'])
print('IGDB token acquired:', bool(token))

## Pull the critic-rated catalogue from IGDB

`fetch_igdb_games()` (in `src/ingest.py`) POSTs Apicalypse queries to `/v4/games`, offset-paged 500 at a time, filtered to `aggregated_rating != null & aggregated_rating_count >= 3`, sorted by critic rating. Genre/platform IDs come back as arrays; we keep them here and resolve to names in the next cell. Every raw page is saved to `data/raw/igdb/`.

In [ ]:
df_raw = fetch_igdb_games(CLIENT_ID, token, cfg)
print(df_raw.shape)
df_raw.head()

## Resolve genre & platform IDs to names
IGDB stores genres/platforms as ID arrays. Resolve them in bulk (two small reference calls) and join comma-separated name strings onto the games table.

In [ ]:
all_genre_ids = sorted({i for ids in df_raw['genre_ids'] for i in ids})
all_platform_ids = sorted({i for ids in df_raw['platform_ids'] for i in ids})

genre_map = _igdb_id_name_map('genres', all_genre_ids, CLIENT_ID, token, cfg)
platform_map = _igdb_id_name_map('platforms', all_platform_ids, CLIENT_ID, token, cfg)
print('genres:', len(genre_map), '| platforms:', len(platform_map))

df_raw['genres'] = df_raw['genre_ids'].apply(lambda ids: ', '.join(genre_map.get(i, '') for i in ids))
df_raw['platforms'] = df_raw['platform_ids'].apply(lambda ids: ', '.join(platform_map.get(i, '') for i in ids))
df_raw = df_raw.drop(columns=['genre_ids', 'platform_ids'])
df_raw.head()

## Load raw to DuckDB + register provenance

In [ ]:
from datetime import date

load_to_duckdb(df_raw, 'igdb_games_raw', con)

register_source(
    con, 'igdb_games_raw',
    name='IGDB — Internet Game Database (API)',
    url='https://api.igdb.com/v4/games',
    license='Free for non-commercial use — Twitch Developer Services Agreement; attribution to IGDB',
    notes=('Games with a critic aggregate backed by >= 3 critic scores '
           '(aggregated_rating_count >= 3). Fields: id, slug, name, release_year, '
           'aggregated_rating (critic 0-100), aggregated_rating_count, rating (user 0-100), '
           'rating_count, total_rating, total_rating_count, genres, platforms. '
           'Raw JSON pages cached in data/raw/igdb/.'),
    retrieved=date.today().isoformat(),
    methodology=('aggregated_rating = IGDB simple mean of external critic outlet scores (0-100); '
                 'rating = IGDB community/user rating (0-100). Two separate measures, not blended '
                 '(total_rating is IGDB\'s blend, kept for reference only). Distinct from Metacritic '
                 '(a different aggregator/method) — describe as IGDB critic rating, not Metacritic.'),
    series_breaks=('IGDB aggregate is an unweighted mean of whatever outlets it has, so composition '
                   'varies by title/era; low aggregated_rating_count aggregates are less stable (>=3 '
                   'floor applied). Scores drift up over time and covered outlets change — cross-era '
                   'comparisons are directional; bin by period. Very recent titles may still be '
                   'accruing critic scores at pull time.'),
)

con.execute('''SELECT COUNT(*) AS n,
  ROUND(MIN(aggregated_rating),1) AS min_c, ROUND(MAX(aggregated_rating),1) AS max_c,
  ROUND(AVG(aggregated_rating),1) AS avg_c,
  SUM(CASE WHEN rating IS NOT NULL THEN 1 ELSE 0 END) AS have_user_rating
  FROM igdb_games_raw''').df()

## Quick inspection

In [ ]:
print('rows:', len(df_raw))
print('with critic aggregate:', df_raw['aggregated_rating'].notna().sum())
print('with user rating:', df_raw['rating'].notna().sum())
yrs = df_raw['release_year'].dropna()
print('release years:', int(yrs.min()), '→', int(yrs.max()))
df_raw['aggregated_rating'].describe()

---
**Next:** `02-clean.ipynb` — clean in DuckDB, cast, derive year/decade, quality report, save interim Parquet.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')